# PFT Flow Execution Chain Report

Run all cells (**Run All**) to generate a detailed report for a `test_pft_flow` run and its child `test_mds_batch_worker` runs.

This notebook reports:
- Parent/child execution chain from `noetl.execution` via `parent_execution_id`
- Event telemetry from `noetl.event` across the full chain
- Command telemetry from `noetl.command` / `noetl.commands` when available
- Facility validation and downstream result-table status from `demo_noetl`

Set `EXECUTION_ID` in the next cell, or leave it empty to auto-detect the latest parent `test_pft_flow` run.

In [ ]:
import psycopg2
import pandas as pd
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ────────────────────────────────────────────────────────────
# Specific parent execution ID to report on (test_pft_flow). Leave empty to auto-detect.
EXECUTION_ID = ""

# Optional safety filter used by auto-detect
MAIN_PLAYBOOK_PATH_HINT = "tests/fixtures/playbooks/pft_flow_test/test_pft_flow"

# noetl control DB (execution, event, command telemetry)
NOETL_HOST     = "722-2.local"
NOETL_PORT     = 54321
NOETL_DBNAME   = "noetl"
NOETL_USER     = "noetl"
NOETL_PASSWORD = "noetl"

# demo_noetl results DB (work queue, validation, fixture tables)
DEMO_HOST     = "722-2.local"
DEMO_PORT     = 54321
DEMO_DBNAME   = "demo_noetl"
DEMO_USER     = "demo"
DEMO_PASSWORD = "demo"
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
conn_noetl = psycopg2.connect(
    host=NOETL_HOST, port=NOETL_PORT,
    dbname=NOETL_DBNAME, user=NOETL_USER, password=NOETL_PASSWORD
)
conn_noetl.autocommit = True

conn_demo = psycopg2.connect(
    host=DEMO_HOST, port=DEMO_PORT,
    dbname=DEMO_DBNAME, user=DEMO_USER, password=DEMO_PASSWORD
)
conn_demo.autocommit = True

print(f"Connected: {NOETL_DBNAME} and {DEMO_DBNAME} @ {NOETL_HOST}:{NOETL_PORT}")


def qn(sql, params=None):
    """Query the noetl control DB."""
    with conn_noetl.cursor() as cur:
        cur.execute(sql, params)
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)


def qd(sql, params=None):
    """Query the demo_noetl results DB."""
    with conn_demo.cursor() as cur:
        cur.execute(sql, params)
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)


def pick_command_table():
    """Return fully qualified command table name when present."""
    probe = qn("""
        SELECT table_schema, table_name
        FROM information_schema.tables
        WHERE table_schema = 'noetl'
          AND table_name IN ('commands', 'command')
        ORDER BY CASE table_name WHEN 'commands' THEN 0 ELSE 1 END
    """)
    if probe.empty:
        return None
    row = probe.iloc[0]
    return f"{row['table_schema']}.{row['table_name']}"


if not EXECUTION_ID:
    df_latest = qn("""
        SELECT e.execution_id, c.path, e.status, e.start_time, e.end_time
        FROM noetl.execution e
        JOIN noetl.catalog c ON c.catalog_id = e.catalog_id
        WHERE c.path = %s
          AND e.parent_execution_id IS NULL
        ORDER BY e.created_at DESC
        LIMIT 10
    """, (MAIN_PLAYBOOK_PATH_HINT,))

    print("Recent parent test_pft_flow executions:")
    display(df_latest)

    if not df_latest.empty:
        EXECUTION_ID = str(df_latest.iloc[0]["execution_id"])
        print(f"\nAuto-selected parent execution_id: {EXECUTION_ID}")
    else:
        print("No matching parent runs found. Set EXECUTION_ID manually.")
else:
    print(f"Using configured parent execution_id: {EXECUTION_ID}")

COMMAND_TABLE = pick_command_table()
print(f"Detected command table: {COMMAND_TABLE or 'not found'}")

EXECUTION_ID = int(EXECUTION_ID) if EXECUTION_ID else None
if EXECUTION_ID is None:
    raise ValueError("EXECUTION_ID is required when auto-detect does not find a run.")

df_chain = qn("""
    WITH RECURSIVE exec_tree AS (
      SELECT
        e.execution_id, e.parent_execution_id, e.catalog_id, e.status,
        e.start_time, e.end_time, e.created_at, 0 AS depth
      FROM noetl.execution e
      WHERE e.execution_id = %s

      UNION ALL

      SELECT
        ch.execution_id, ch.parent_execution_id, ch.catalog_id, ch.status,
        ch.start_time, ch.end_time, ch.created_at, et.depth + 1
      FROM noetl.execution ch
      JOIN exec_tree et ON ch.parent_execution_id = et.execution_id
    )
    SELECT
      et.depth, et.execution_id, et.parent_execution_id, et.status,
      c.path AS playbook_path, et.start_time, et.end_time, et.created_at,
      ROUND(EXTRACT(EPOCH FROM (COALESCE(et.end_time, NOW()) - et.start_time))::numeric, 3) AS duration_s
    FROM exec_tree et
    LEFT JOIN noetl.catalog c ON c.catalog_id = et.catalog_id
    ORDER BY et.depth, et.start_time NULLS LAST, et.execution_id
""", (EXECUTION_ID,))

if df_chain.empty:
    raise ValueError(f"Execution {EXECUTION_ID} not found in noetl.execution")

CHAIN_EXEC_IDS = df_chain["execution_id"].astype("int64").tolist()
print(f"Execution chain size: {len(CHAIN_EXEC_IDS)} (parent + descendants)")

## Execution Chain Overview

Parent + descendant executions, where descendants are linked through `parent_execution_id`.

In [ ]:
display(df_chain)

parent_row = df_chain.loc[df_chain["execution_id"] == EXECUTION_ID].iloc[0]
child_count = max(len(df_chain) - 1, 0)

print(f"Parent execution_id: {EXECUTION_ID}")
print(f"Parent playbook    : {parent_row['playbook_path']}")
print(f"Parent status      : {parent_row['status']}")
print(f"Total child runs   : {child_count}")

if child_count:
    child_paths = (
        df_chain[df_chain["execution_id"] != EXECUTION_ID]
        .groupby("playbook_path", dropna=False)["execution_id"]
        .count()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    print("\nChild playbooks observed:")
    display(child_paths)

## Event Coverage Across Chain

Event counts for parent and child runs combined, grouped by execution and event type.

In [ ]:
df_ev_counts = qn("""
    SELECT
        e.execution_id,
        c.path AS playbook_path,
        ev.event_type,
        COUNT(*) AS event_count,
        ROUND(AVG(ev.duration)::numeric, 3) AS avg_duration_s
    FROM noetl.event ev
    JOIN noetl.execution e ON e.execution_id = ev.execution_id
    LEFT JOIN noetl.catalog c ON c.catalog_id = e.catalog_id
    WHERE ev.execution_id = ANY(%s::bigint[])
    GROUP BY e.execution_id, c.path, ev.event_type
    ORDER BY e.execution_id, ev.event_type
""", (CHAIN_EXEC_IDS,))

display(df_ev_counts)

summary = qn("""
    SELECT
        COUNT(*) FILTER (WHERE event_type = 'command.issued') AS command_issued,
        COUNT(*) FILTER (WHERE event_type = 'command.started') AS command_started,
        COUNT(*) FILTER (WHERE event_type = 'command.completed') AS command_completed,
        COUNT(*) FILTER (WHERE event_type = 'call.done') AS call_done,
        COUNT(*) FILTER (WHERE event_type = 'call.error') AS call_error,
        COUNT(*) FILTER (WHERE event_type = 'loop.done') AS loop_done
    FROM noetl.event
    WHERE execution_id = ANY(%s::bigint[])
""", (CHAIN_EXEC_IDS,))

print("Chain-level event summary:")
display(summary)

## Command Table Telemetry

Uses `noetl.commands` (or `noetl.command`) when present.

In [ ]:
if COMMAND_TABLE is None:
    print("Command table not found in noetl schema. Skipping command-table section.")
else:
    schema_name, table_name = COMMAND_TABLE.split('.')
    df_cols = qn("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = %s AND table_name = %s
    """, (schema_name, table_name))
    cols = set(df_cols["column_name"].tolist())

    duration_expr = (
        "duration_ms" if "duration_ms" in cols
        else "(EXTRACT(EPOCH FROM (completed_at - started_at)) * 1000)"
        if {"started_at", "completed_at"}.issubset(cols)
        else "NULL"
    )
    status_expr = "status" if "status" in cols else "NULL::text"
    tool_expr = "tool_kind" if "tool_kind" in cols else "NULL::text"

    sql_status = f"""
        SELECT
            execution_id,
            {status_expr} AS status,
            COUNT(*) AS command_count,
            ROUND(AVG({duration_expr})::numeric, 2) AS avg_duration_ms,
            ROUND((percentile_cont(0.95) WITHIN GROUP (ORDER BY {duration_expr}))::numeric, 2) AS p95_duration_ms
        FROM {COMMAND_TABLE}
        WHERE execution_id = ANY(%s::bigint[])
        GROUP BY execution_id, {status_expr}
        ORDER BY execution_id, {status_expr}
    """
    df_cmd = qn(sql_status, (CHAIN_EXEC_IDS,))
    display(df_cmd)

    sql_tool = f"""
        SELECT
            execution_id,
            COALESCE({tool_expr}, 'unknown') AS tool_kind,
            COUNT(*) AS command_count
        FROM {COMMAND_TABLE}
        WHERE execution_id = ANY(%s::bigint[])
        GROUP BY execution_id, COALESCE({tool_expr}, 'unknown')
        ORDER BY execution_id, command_count DESC
    """
    df_cmd_type = qn(sql_tool, (CHAIN_EXEC_IDS,))

    print("\nCommands by tool kind:")
    display(df_cmd_type)

## Facility Validation Log (Parent Run)

Rows written by `validate_facility_results` in the parent execution.

In [ ]:
df_v = qd("""
    SELECT
        execution_id,
        facility_mapping_id,
        assessments_done,
        conditions_done,
        medications_done,
        vital_signs_done,
        demographics_done,
        assessments_queue_done,
        total_expected,
        logged_at
    FROM public.pft_test_validation_log
    WHERE execution_id = %s
    ORDER BY facility_mapping_id
""", (EXECUTION_ID,))

DATA_COLS = [
    "assessments_done", "conditions_done", "medications_done",
    "vital_signs_done", "demographics_done", "assessments_queue_done",
]

if df_v.empty:
    print("No validation rows yet for parent execution.")
else:
    def highlight_val(val):
        if pd.isna(val):
            return "background-color: #fff3cd"
        return (
            "background-color: #d4edda; color: #155724" if int(val) == 1000
            else "background-color: #f8d7da; color: #721c24"
        )

    display(df_v.style.applymap(highlight_val, subset=DATA_COLS))

    n_done = len(df_v)
    all_pass = all(df_v[c].fillna(0).astype(int).eq(1000).all() for c in DATA_COLS)
    print(f"\nFacilities validated: {n_done} / 10")
    print("GO" if all_pass and n_done == 10 else "NO-GO")

## Recent Events Across Parent + Child Runs

Most recent chain events to inspect stuck phases, retries, and failures.

In [ ]:
df_recent = qn("""
    SELECT
        ev.event_id,
        ev.execution_id,
        c.path AS playbook_path,
        ev.event_type,
        ev.node_name,
        ev.status,
        ROUND(ev.duration::numeric, 3) AS duration_s,
        ev.created_at,
        ev.error
    FROM noetl.event ev
    JOIN noetl.execution e ON e.execution_id = ev.execution_id
    LEFT JOIN noetl.catalog c ON c.catalog_id = e.catalog_id
    WHERE ev.execution_id = ANY(%s::bigint[])
    ORDER BY ev.event_id DESC
    LIMIT 200
""", (CHAIN_EXEC_IDS,))

print(f"Recent events returned: {len(df_recent)}")
display(df_recent)

## Command Lifecycle Timing from Events

Computes per-command runtime from `command.started` to `command.completed` using `meta.command_id`.

In [ ]:
df_cmd_lifecycle = qn("""
    WITH started AS (
      SELECT execution_id, meta->>'command_id' AS command_id, MIN(created_at) AS started_at
      FROM noetl.event
      WHERE execution_id = ANY(%s::bigint[])
        AND event_type = 'command.started'
        AND meta ? 'command_id'
      GROUP BY execution_id, meta->>'command_id'
    ),
    completed AS (
      SELECT execution_id, meta->>'command_id' AS command_id, MAX(created_at) AS completed_at
      FROM noetl.event
      WHERE execution_id = ANY(%s::bigint[])
        AND event_type = 'command.completed'
        AND meta ? 'command_id'
      GROUP BY execution_id, meta->>'command_id'
    )
    SELECT
      s.execution_id,
      s.command_id,
      ROUND((EXTRACT(EPOCH FROM (c.completed_at - s.started_at)) * 1000)::numeric, 2) AS runtime_ms
    FROM started s
    JOIN completed c
      ON c.execution_id = s.execution_id
     AND c.command_id = s.command_id
    ORDER BY s.execution_id, runtime_ms DESC
""", (CHAIN_EXEC_IDS, CHAIN_EXEC_IDS))

display(df_cmd_lifecycle.head(200))

if not df_cmd_lifecycle.empty:
    agg = (
        df_cmd_lifecycle.groupby("execution_id")["runtime_ms"]
        .agg(["count", "mean", "max"])
        .reset_index()
        .rename(columns={"count": "commands", "mean": "avg_ms", "max": "max_ms"})
    )
    print("\nExecution-level lifecycle summary:")
    display(agg)

## Result Table Snapshot

Quick table-size and queue-state checks in `demo_noetl`.

In [ ]:
RESULT_TABLES = [
    "pft_test_patient_assessments",
    "pft_test_patient_conditions",
    "pft_test_patient_medications",
    "pft_test_patient_vital_signs",
    "pft_test_patient_demographics",
    "pft_test_mds_assessment_ids_work",
    "pft_test_mds_assessment_details",
]

rows = []
for table_name in RESULT_TABLES:
    df_cnt = qd(f"SELECT COUNT(*) AS cnt FROM public.{table_name}")
    rows.append({"table": table_name, "row_count": int(df_cnt.iloc[0]["cnt"])})

display(pd.DataFrame(rows))

df_queue = qd("""
    SELECT
        data_type,
        COUNT(*) FILTER (WHERE status = 'done') AS done_rows,
        COUNT(*) FILTER (WHERE status = 'claimed') AS claimed_rows,
        COUNT(*) FILTER (WHERE status = 'pending') AS pending_rows,
        COUNT(*) FILTER (WHERE status = 'error') AS error_rows,
        COUNT(*) AS total_rows
    FROM public.pft_test_patient_work_queue
    WHERE execution_id = %s
    GROUP BY data_type
    ORDER BY data_type
""", (str(EXECUTION_ID),))

print("\nWork queue status for parent execution:")
display(df_queue)

## Final Verdict Table

Quick triage view per execution in the chain: status, command totals, call errors, and last event timestamp.

In [ ]:
df_verdict = qn("""
    WITH ev AS (
        SELECT
            execution_id,
            MAX(created_at) AS last_event_at,
            COUNT(*) FILTER (WHERE event_type = 'call.error') AS call_error_count,
            COUNT(*) FILTER (WHERE event_type = 'command.issued') AS command_issued_count,
            COUNT(*) FILTER (WHERE event_type = 'command.completed') AS command_completed_count
        FROM noetl.event
        WHERE execution_id = ANY(%s::bigint[])
        GROUP BY execution_id
    ),
    child_counts AS (
        SELECT
            parent_execution_id AS execution_id,
            COUNT(*) AS child_execution_count
        FROM noetl.execution
        WHERE parent_execution_id = ANY(%s::bigint[])
        GROUP BY parent_execution_id
    )
    SELECT
        e.execution_id,
        CASE WHEN e.execution_id = %s THEN 'parent' ELSE 'child' END AS role,
        c.path AS playbook_path,
        e.status,
        COALESCE(cc.child_execution_count, 0) AS child_execution_count,
        COALESCE(ev.command_issued_count, 0) AS command_issued_count,
        COALESCE(ev.command_completed_count, 0) AS command_completed_count,
        COALESCE(ev.call_error_count, 0) AS call_error_count,
        ev.last_event_at,
        ROUND(EXTRACT(EPOCH FROM (COALESCE(e.end_time, NOW()) - e.start_time))::numeric, 3) AS duration_s
    FROM noetl.execution e
    LEFT JOIN noetl.catalog c ON c.catalog_id = e.catalog_id
    LEFT JOIN ev ON ev.execution_id = e.execution_id
    LEFT JOIN child_counts cc ON cc.execution_id = e.execution_id
    WHERE e.execution_id = ANY(%s::bigint[])
    ORDER BY role DESC, e.execution_id
""", (CHAIN_EXEC_IDS, CHAIN_EXEC_IDS, EXECUTION_ID, CHAIN_EXEC_IDS))

display(df_verdict)

if not df_verdict.empty:
    parent = df_verdict[df_verdict["role"] == "parent"].iloc[0]
    no_errors = int(parent["call_error_count"]) == 0
    completed = str(parent["status"]).upper() in {"COMPLETED", "SUCCESS"}
    verdict = "PASS" if completed and no_errors else "CHECK"
    print(f"Parent verdict: {verdict}")
    print(f"Parent status : {parent['status']}")
    print(f"Call errors   : {int(parent['call_error_count'])}")
    print(f"Children      : {int(parent['child_execution_count'])}")

## Close Connections

In [ ]:
conn_noetl.close()
conn_demo.close()
print("Connections closed.")